In [2]:
import subprocess
import os
import re

class Fermat:
    """
    Interface to the Fermat computer algebra system from SageMath.
    
    This class provides a way to send commands to Fermat and retrieve results.
    """
    
    def __init__(self, fermat_path=None, timeout=10, workdir=None):
        """
        Initialize the Fermat interface.
        
        Parameters:
        - fermat_path: Path to the Fermat executable. If None, tries to find it in PATH.
        - timeout: Maximum time to wait for a response from Fermat (in seconds).
        """
        self.timeout = timeout
        self.last_cpu_time = 0.0
        self.total_cpu_time = 0.0
        
        if fermat_path is None:
            # Try to find Fermat in the system PATH
            self.fermat_path = self._find_fermat()
        else:
            self.fermat_path = fermat_path
            
        if not self.fermat_path:
            raise RuntimeError("Fermat executable not found. Please specify the path to Fermat.")
        self.workdir = os.path.abspath(workdir or os.path.dirname(self.fermat_path) or os.getcwd())
    
    def _find_fermat(self):
        """Try to locate the Fermat executable in the system PATH."""
        for path in os.environ["PATH"].split(os.pathsep):
            exe_path = os.path.join(path, "fer64")
            if os.path.isfile(exe_path) and os.access(exe_path, os.X_OK):
                return exe_path
            exe_path = os.path.join(path, "fermat")
            if os.path.isfile(exe_path) and os.access(exe_path, os.X_OK):
                return exe_path
        return None
    
    def _clean_output(self, output):
        cpu_times = [float(value) for value in re.findall(
            r'Elapsed CPU time:\s*([0-9]+(?:\.[0-9]+)?)', output
        )]
        self.last_cpu_time = sum(cpu_times)
        self.total_cpu_time += self.last_cpu_time
        return output.strip()

    def cpu_time(self):
        return self.total_cpu_time
    
    def execute(self, command):
        """
        Execute a command in Fermat and return the result as a string.
        
        Parameters:
        - command: The Fermat command to execute (e.g., "1+1")
        
        Returns:
        - The output from Fermat as a string
        """
        # Prepare the command to exit after execution
        full_command = f"{command}\n&q\n"
        
        try:
            # Start Fermat process
            proc = subprocess.Popen(
                [self.fermat_path],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                cwd=self.workdir,
                text=True
            )

            # Send command and get output
            stdout, stderr = proc.communicate(
                input=full_command,
                timeout=self.timeout
            )
            
            if proc.returncode != 0:
                raise RuntimeError(f"Fermat returned error code {proc.returncode}: {stderr}")
            
            # Clean and return the output
            return self._clean_output(stdout)
            
        except subprocess.TimeoutExpired:
            proc.kill()
            raise RuntimeError("Fermat computation timed out")
        except Exception as e:
            raise RuntimeError(f"Error communicating with Fermat: {str(e)}")
    
    def __call__(self, command):
        """Alias for execute() to allow fermat("command") syntax."""
        return self.execute(command)
    
    # Short alias
    exc = execute


# Example usage:
if __name__ == "__main__":
    # Create a Fermat interface instance
    fermat = Fermat("/home/ideal/drsolve/Ferl7/fer64", timeout=120)
    # Execute a simple command
    '''
    result = fermat.exc("1+1")
    print(f"1+1 in Fermat is: {result}")
    
    # You can also use the __call__ syntax
    result = fermat("2^100")
    print(f"2^100 in Fermat is: {result}")
    '''
    result = fermat("&(R = 'cryp');&(R = '3*3');")
    #print(f"cryp in Fermat is: {result}")
    print(fermat.cpu_time())

0.0950000000000000


In [4]:
from itertools import product
from pathlib import Path
import csv
import random
import time

def _dense_exponents(n, degree):
    """Return every exponent vector with total degree at most degree."""
    exponents = []
    for exponent in product(range(degree + 1), repeat=n):
        if sum(exponent) <= degree:
            exponents.append(exponent)
    return exponents

def _fermat_monomial(exponent, variables):
    factors = []
    for variable, power in zip(variables, exponent):
        if power == 1:
            factors.append(variable)
        elif power > 1:
            factors.append(f"{variable}^{power}")
    return "*".join(factors) or "1"

def _first_primes(count):
    primes = []
    candidate = 2
    while len(primes) < count:
        is_prime = candidate >= 2 and all(
            candidate % divisor for divisor in range(2, int(candidate ** 0.5) + 1)
        )
        if is_prime:
            primes.append(candidate)
        candidate += 1
    return primes

def generate_dense_system(n, degree, prime=536870923, seed=None):
    """Generate n dense polynomials in x0,...,x(n-1) over F_prime."""
    if n < 2 or degree < 1:
        raise ValueError("n must be >= 2 and degree must be >= 1")
    rng = random.Random(seed)
    variables = [f"zq{i}" for i in range(n)]
    monomials = [_fermat_monomial(e, variables) for e in _dense_exponents(n, degree)]
    polynomials = []
    for _ in range(n):
        terms = [f"{rng.randrange(1, prime)}*{monomial}" for monomial in monomials]
        polynomials.append(" + ".join(terms))
    return polynomials

def write_fermat_case(n, degree, workdir, prime=536870923, seed=None, template_name="cryp"):
    """Write a standalone case file using the current 3*3 driver."""
    root = Path(workdir).resolve()
    template = (root / template_name).read_text()
    function_start = template.index(";; Dixon routines from 2018.")
    function_end = template.index(";;**  arrays")
    functions = template[function_start:function_end]
    variables = [f"zq{i}" for i in range(n)]
    declarations = "\n".join(f"&(J = {name});" for name in variables)
    equations = generate_dense_system(n, degree, prime=prime, seed=seed)
    body = ",\n          ".join(equations)
    retained = variables[0]
    samples = ",".join(map(str, _first_primes(n)))
    driver = (
        "Array num[10000],den[10000],lcm[1500];\n"
        "nn:=0; cnt:=0; zer:=0; one:=1;\n"
        "Ma([d]);\n"
        f"[m3] := [m2]#({retained} = {samples});\n"
        "Pseudet([m3],[c2]);\n"
        "Array m4[Cols[c2], Cols[c2]];\n"
        "&(D=-1);\n!!(Fill);\n"
        "s := Det[m4];\n&x;\n"
    )
    case = (
        "&(p = {prime});\n"
        "{declarations}\n"
        "@([d]);\nArray d[{n}];\n[d] := [[{body}]];\n\n"
        "{functions}\n"
        "{driver}"
    ).format(prime=prime, declarations=declarations, n=n, body=body, functions=functions, driver=driver)
    path = root / f"dense_{n}x{degree}_seed{seed}.fer"
    path.write_text(case)
    return path

def run_dense_case(n, degree, fermat_path="/home/ideal/drsolve/Ferl7/fer64",
                   workdir="/home/ideal/drsolve/Ferl7", prime=536870923, seed=1, timeout=600):
    start = time.perf_counter()
    case_path = write_fermat_case(n, degree, workdir, prime=prime, seed=seed)
    generation_time = time.perf_counter() - start
    fermat = Fermat(fermat_path, timeout=timeout, workdir=workdir)
    try:
        output = fermat(f"&(R = '{case_path.name}');")
        status = "ok"
    except Exception as error:
        output = str(error)
        status = "error"
    finally:
        wall_time = time.perf_counter() - start
    return {
        "n": n, "degree": degree, "seed": seed,
        "status": status, "generation_seconds": generation_time,
        "cpu_seconds": fermat.total_cpu_time,
        "wall_seconds": wall_time, "case": str(case_path),
        "output": output,
    }

def run_dense_benchmark(fermat_path="/home/ideal/drsolve/Ferl7/fer64",
                        workdir="/home/ideal/drsolve/Ferl7", prime=536870923,
                        seed=1, timeout=600, save_csv="dense_benchmark.csv"):
    sizes = [(3, degree) for degree in range(2, 13)]
    sizes += [(4, degree) for degree in range(2, 7)]
    sizes += [(5, degree) for degree in range(2, 5)]
    sizes += [(6, degree) for degree in range(2, 4)]
    sizes += [(7, degree) for degree in range(2, 3)]
    results = []
    for n, degree in sizes:
        print(f"Running {n}x{degree}...", flush=True)
        result = run_dense_case(n, degree, fermat_path=fermat_path,
                                workdir=workdir, prime=prime, seed=seed,
                                timeout=timeout)
        results.append(result)
        print(f"  {result['status']}, "
              f"time={result['wall_seconds']:.6f}s")
    fields = ["n", "degree", "seed", "status", "generation_seconds",
              "cpu_seconds", "wall_seconds", "case"]
    csv_path = Path(workdir) / save_csv
    with csv_path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows({key: result[key] for key in fields} for result in results)
    return results

# Run all requested sizes with a reproducible random system:
results = run_dense_benchmark(seed=20260828, timeout=3600)


Running 3x2...
  ok, time=0.108246s
Running 3x3...
  ok, time=0.129496s
Running 3x4...
  ok, time=0.202786s
Running 3x5...
  ok, time=0.470568s
Running 3x6...
  ok, time=1.842786s
Running 3x7...
  ok, time=8.044608s
Running 3x8...
  ok, time=31.799852s
Running 3x9...
  ok, time=121.017261s
Running 3x10...
  ok, time=410.290264s
Running 3x11...
  ok, time=1359.458466s
Running 3x12...
  error, time=3600.130859s
Running 4x2...
  ok, time=0.123112s
Running 4x3...
  ok, time=0.593351s
Running 4x4...
  ok, time=10.683009s
Running 4x5...
  ok, time=270.129372s
Running 4x6...
  error, time=3600.131998s
Running 5x2...
  ok, time=0.510635s
Running 5x3...
  ok, time=31.475841s
Running 5x4...
  error, time=3600.080489s
Running 6x2...
  ok, time=12.398708s
Running 6x3...
  error, time=3600.111534s
Running 7x2...
  error, time=3600.110870s
